In [2]:
pip install pinecone openai tqdm

     ---------------------------------------- 0.0/43.1 kB ? eta -:--:--
     ---------------------------------------- 43.1/43.1 kB 2.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   - -------------------------------------- 0.1/2.6 MB 3.6 MB/s eta 0:00:01
   ----- ---------------------------------- 0.4/2.6 MB 3.8 MB/s eta 0:00:01
   ------- -------------------------------- 0.5/2.6 MB 3.7 MB/s eta 0:00:01
   ---------- ----------------------------- 0.7/2.6 MB 4.2 MB/s eta 0:00:01
   ------------ --------------------------- 0.8/2.6 MB 3.8 MB/s eta 0:00:01
   -------------- ------------------------- 1.0/2.6 MB 3.8 MB/s eta 0:00:01
   ----------------- ---------------------- 1.2/2.6 MB 3.9 MB/s eta 0:00:01
   --------------------- ------------------ 1.4/2.6 MB 4.2 MB/s eta 0:00:01
   ------------------------- -------------- 1.6/2.6 MB 4.4 MB/s eta 0:00:01
   ---------------------------- ----------- 1.8/2.6 MB 4.1 MB/s eta 0:00:01
   ----------------


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
import os

load_dotenv()

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index_name = "agent-rag"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536,  # OpenAI embedding dimension
        metric="dotproduct",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print("Index ready")

Index ready


In [35]:
from openai import OpenAI
from pinecone import Pinecone
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

index = pc.Index("agent-rag")

In [36]:
def get_dense_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    return response.data[0].embedding

def get_sparse_embedding(text):

    response = pc.inference.embed(
        model="pinecone-sparse-english-v0",
        inputs=[text],
        parameters={
            "input_type": "passage"
        }
    )

    sparse = response.data[0]

    return {
        "indices": sparse["sparse_indices"],
        "values": sparse["sparse_values"]
    }

In [37]:
def clean_metadata(metadata, text):

    origin = metadata.get("origin")

    # Ensure origin is a dict
    if not isinstance(origin, dict):
        origin = {}

    return {

        "filename": str(metadata.get("filename") or ""),

        "chunk_id": int(metadata.get("chunk_id") or 0),

        "title": str(metadata.get("title") or ""),

        "token_count": int(metadata.get("token_count") or 0),

        # Pinecone supports list[str]
        "page_numbers": [
            str(p)
            for p in metadata.get("page_numbers", [])
        ],

        "headings": [
            str(h)
            for h in metadata.get("headings", [])
        ],

        "captions": [
            str(c)
            for c in metadata.get("captions", [])
        ],

        "doc_items": [
            str(d)
            for d in metadata.get("doc_items", [])
        ],

        # Flatten origin safely
        "origin_filename": str(origin.get("filename") or ""),

        "origin_mimetype": str(origin.get("mimetype") or ""),

        "origin_binary_hash": str(
            origin.get("binary_hash") or ""
        ),

        # Store actual chunk text
        "text": str(text)
    }


In [38]:
import json
from tqdm import tqdm
BATCH_SIZE=10
chunk_file=r"data/chunks/all_chunks.json"
with open(chunk_file, "r", encoding="utf-8") as f:
        chunks = json.load(f)
batch = []
for chunk in tqdm(chunks):

        try:

            text = chunk["text"]
            metadata = chunk["metadata"]
            print(type(metadata))
            # -------------------------
            # Dense Embedding
            # -------------------------

            dense_vector = get_dense_embedding(text)

            # -------------------------
            # Sparse Embedding
            # -------------------------

            sparse_vector = get_sparse_embedding(text)

            # -------------------------
            # Vector Object
            # -------------------------
            cleaned_metadata = clean_metadata(metadata, text)
            vector = {
                "id": f"{metadata['filename']}_{metadata['chunk_id']}",

                "values": dense_vector,

                "sparse_values": sparse_vector,

                "metadata": cleaned_metadata
            }

            batch.append(vector)

            # -------------------------
            # Batch Upload
            # -------------------------

            if len(batch) >= BATCH_SIZE:

                index.upsert(vectors=batch)

                print(f"Uploaded batch of {len(batch)}")

                batch = []

        except Exception as e:

            print(f"Error processing chunk: {e}")

# =========================
# FINAL UPLOAD
# =========================

if batch:

    index.upsert(vectors=batch)

    print(f"Uploaded final batch of {len(batch)}")

print("\nUpload Complete")

  0%|          | 0/183 [00:00<?, ?it/s]

<class 'dict'>


  1%|          | 1/183 [00:01<05:42,  1.88s/it]

<class 'dict'>


  1%|          | 2/183 [00:03<05:25,  1.80s/it]

...

<class 'dict'>


 99%|█████████▉| 182/183 [03:23<00:01,  1.19s/it]

<class 'dict'>


100%|██████████| 183/183 [03:24<00:00,  1.12s/it]


Uploaded final batch of 3

Upload Complete
